# LeetCode #122 — Best Time to Buy and Sell Stock II: tham lam cộng mọi đoạn tăng, và DP trạng thái

**Mức độ:** Medium · **Chủ đề:** Array, Dynamic Programming, Greedy

Skeleton trên LeetCode:

```python
class Solution:
    def maxProfit(self, prices: List[int]) -> int:
```

Mục tiêu của notebook:

1. Cách nhìn **tham lam**: lợi nhuận tối đa = tổng mọi bước **tăng giá** `max(0, p[i] − p[i−1])`.
2. Chứng minh vì sao tham lam đúng (phân rã một giao dịch dài thành các bước một ngày).
3. Cách nhìn **DP hai trạng thái** `hold` / `cash` — khuôn mẫu dùng cho cả họ bài cổ phiếu (#121, #123, #188, #309, #714).
4. Đếm số giao dịch thực sự và in ra các cặp (mua, bán).

# 1. Bài toán là gì?

Cho mảng `prices`, `prices[i]` là giá cổ phiếu ngày `i`. Mỗi ngày bạn có thể mua và/hoặc bán. Bạn chỉ được **giữ tối đa một cổ phiếu** tại một thời điểm, nhưng có thể mua rồi bán **ngay trong cùng ngày**.

Tìm **lợi nhuận lớn nhất**.

**Constraints (theo đề):**

- `1 <= prices.length <= 3 * 10^4`
- `0 <= prices[i] <= 10^4`

---

## Ví dụ

```text
[7, 1, 5, 3, 6, 4]  ->  7     mua 1 bán 5 (+4), mua 3 bán 6 (+3)
[1, 2, 3, 4, 5]     ->  4     mua 1 bán 5 (+4)  (= +1 +1 +1 +1)
[7, 6, 4, 3, 1]     ->  0     không giao dịch
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Thuật toán tham lam (greedy)
2. Quy hoạch động (Dynamic Programming — DP)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Thuật toán tham lam (greedy)

### Tham lam là gì?

Thuật toán tham lam xây lời giải bằng cách ở **mỗi bước chọn phương án tốt nhất trước mắt**, không bao giờ quay lại sửa. Nhanh và đơn giản — nhưng **không phải lúc nào cũng đúng**.

### Ví dụ đúng: đổi tiền với mệnh giá 1, 2, 5, 10

Luôn lấy tờ **lớn nhất** không vượt số tiền còn lại.

### Ví dụ sai: mệnh giá 1, 3, 4, cần 6

Tham lam: `4 + 1 + 1` (3 tờ). Tối ưu: `3 + 3` (2 tờ). Chọn tốt nhất trước mắt dẫn tới kết quả tệ hơn.

In [1]:
def doi_tien_tham_lam(menh_gia, so_tien):
    chon = []
    for m in sorted(menh_gia, reverse=True):
        while so_tien >= m:
            so_tien -= m
            chon.append(m)
    return chon

def doi_tien_toi_uu(menh_gia, so_tien):           # DP để đối chiếu
    INF = float("inf")
    dp = [0] + [INF] * so_tien
    for x in range(1, so_tien + 1):
        for m in menh_gia:
            if m <= x:
                dp[x] = min(dp[x], dp[x - m] + 1)
    return dp[so_tien]

print("mệnh giá [1,2,5,10], 18:", doi_tien_tham_lam([1, 2, 5, 10], 18), "| tối ưu:", doi_tien_toi_uu([1, 2, 5, 10], 18), "tờ")
print("mệnh giá [1,3,4], 6:", doi_tien_tham_lam([1, 3, 4], 6), "| tối ưu:", doi_tien_toi_uu([1, 3, 4], 6), "tờ")

mệnh giá [1,2,5,10], 18: [10, 5, 2, 1] | tối ưu: 4 tờ
mệnh giá [1,3,4], 6: [4, 1, 1] | tối ưu: 2 tờ


### Làm sao biết tham lam đúng?

Phải **chứng minh**. Hai kỹ thuật hay dùng:

1. **Lập luận trao đổi (exchange argument)**: giả sử có lời giải tối ưu khác lựa chọn tham lam ở bước đầu; chỉ ra có thể **đổi** nó sang lựa chọn tham lam mà không tệ đi.
2. **Bất biến "luôn dẫn đầu"**: sau mỗi bước, lời giải tham lam không kém bất kỳ lời giải nào khác.

Và luôn **thử phản ví dụ** bằng brute force trên dữ liệu nhỏ.

### Ví dụ kinh điển: chọn nhiều cuộc họp nhất không trùng giờ

Chọn cuộc họp **kết thúc sớm nhất** trước — để lại nhiều thời gian nhất cho phần còn lại (lập luận trao đổi: thay cuộc họp đầu của lời giải tối ưu bằng cuộc kết thúc sớm nhất vẫn không chồng lấn).

In [2]:
def nhieu_cuoc_hop(hop):
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop, key=lambda x: x[1]):     # theo giờ kết thúc
        if s >= ket_thuc:
            chon.append((s, e))
            ket_thuc = e
    return chon

hop_demo = [(1, 4), (3, 5), (0, 6), (5, 7), (3, 9), (5, 9), (6, 10), (8, 11), (8, 12), (2, 14), (12, 16)]
def theo_bat_dau(hop):                               # tiêu chí SAI để so sánh
    chon, ket_thuc = [], float("-inf")
    for s, e in sorted(hop):
        if s >= ket_thuc:
            chon.append((s, e)); ket_thuc = e
    return chon

print("kết thúc sớm nhất trước:", nhieu_cuoc_hop(hop_demo), "→", len(nhieu_cuoc_hop(hop_demo)), "cuộc")
print("bắt đầu sớm nhất trước (sai):", theo_bat_dau(hop_demo), "→", len(theo_bat_dau(hop_demo)), "cuộc")

kết thúc sớm nhất trước: [(1, 4), (5, 7), (8, 11), (12, 16)] → 4 cuộc
bắt đầu sớm nhất trước (sai): [(0, 6), (6, 10), (12, 16)] → 3 cuộc


### Lỗi hay gặp

- Tin tham lam đúng mà không chứng minh / không thử phản ví dụ.
- Chọn sai tiêu chí tham lam (bắt đầu sớm nhất, ngắn nhất… thay vì kết thúc sớm nhất).
- Quên sắp xếp theo tiêu chí trước khi duyệt.

## 2.2 Quy hoạch động (Dynamic Programming — DP)

### Khi nào dùng DP?

Quy hoạch động dùng cho bài toán có hai đặc điểm:

1. **Cấu trúc con tối ưu**: đáp án của bài lớn được ghép từ đáp án của các **bài con** nhỏ hơn.
2. **Bài con chồng lấp**: cùng một bài con xuất hiện **nhiều lần** nếu đệ quy thẳng.

Thay vì tính lại, ta **lưu** đáp án mỗi bài con vào bảng và dùng lại → mỗi bài con tính đúng một lần.

### Bốn câu hỏi khi thiết kế DP

1. **Trạng thái**: `dp[i]` nghĩa là gì? (viết thành câu tiếng Việt rõ ràng)
2. **Chuyển trạng thái**: `dp[i]` tính từ những `dp[j]` nào?
3. **Cơ sở**: giá trị nhỏ nhất biết sẵn (`dp[0]`, `dp[1]`…).
4. **Đáp án**: là `dp[n]`, hay `max(dp)`, hay …?

### Ví dụ: leo cầu thang

Mỗi bước leo 1 hoặc 2 bậc. Có bao nhiêu cách leo `n` bậc?

- Trạng thái: `dp[i]` = số cách lên tới bậc `i`.
- Chuyển: bước cuối cùng là 1 bậc (từ `i−1`) hoặc 2 bậc (từ `i−2`) → `dp[i] = dp[i−1] + dp[i−2]`.
- Cơ sở: `dp[0] = 1` (đứng yên), `dp[1] = 1`.

In [3]:
from functools import lru_cache

# Cách 1: đệ quy thuần — chậm vì tính lặp
def leo_de_quy(n):
    return 1 if n <= 1 else leo_de_quy(n - 1) + leo_de_quy(n - 2)

# Cách 2: đệ quy + ghi nhớ (top-down, memoization)
@lru_cache(maxsize=None)
def leo_memo(n):
    return 1 if n <= 1 else leo_memo(n - 1) + leo_memo(n - 2)

# Cách 3: điền bảng từ nhỏ tới lớn (bottom-up, tabulation)
def leo_bang(n):
    dp = [0] * (n + 1)
    dp[0] = 1
    if n >= 1:
        dp[1] = 1
    for i in range(2, n + 1):
        dp[i] = dp[i - 1] + dp[i - 2]
    return dp[n], dp

# Cách 4: chỉ giữ hai giá trị gần nhất — O(1) bộ nhớ
def leo_gon(n):
    a, b = 1, 1                 # dp[i-2], dp[i-1]
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b

print("n = 10:", leo_de_quy(10), leo_memo(10), leo_bang(10)[0], leo_gon(10))
print("bảng dp:", leo_bang(10)[1])

n = 10: 89 89 89 89
bảng dp: [1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Top-down hay bottom-up?

| | Top-down (memo) | Bottom-up (bảng) |
|---|---|---|
| Viết | gần với công thức đệ quy, dễ nghĩ | phải chọn **thứ tự** điền bảng |
| Chỉ tính bài con cần thiết | có | không (tính hết) |
| Rủi ro | độ sâu đệ quy | không |
| Tối ưu bộ nhớ | khó | dễ (giữ vài hàng/ô gần nhất) |

### Ví dụ thứ hai: chọn hay không chọn (House Robber)

Dãy nhà có tiền `a[i]`, không được lấy hai nhà **liền nhau**. Lấy tối đa bao nhiêu?

- `dp[i]` = số tiền tối đa khi chỉ xét `i` nhà đầu.
- Nhà thứ `i` (chỉ số `i−1`): **không lấy** → `dp[i−1]`; **lấy** → `dp[i−2] + a[i−1]`. Lấy `max`.

In [4]:
def ke_trom(a):
    dp = [0] * (len(a) + 1)
    if a:
        dp[1] = a[0]
    for i in range(2, len(a) + 1):
        dp[i] = max(dp[i - 1], dp[i - 2] + a[i - 1])
    return dp[-1], dp

print("ke_trom([2, 7, 9, 3, 1]) =", ke_trom([2, 7, 9, 3, 1]))

ke_trom([2, 7, 9, 3, 1]) = (12, [0, 2, 7, 11, 11, 12])


### Lỗi hay gặp

- Định nghĩa trạng thái mơ hồ → công thức sai. Luôn viết `dp[i] = …` bằng lời trước.
- Lệch chỉ số giữa `dp` (dài `n+1`) và mảng đầu vào (dài `n`).
- Điền bảng sai thứ tự (dùng ô chưa tính).
- Quên cơ sở hoặc khởi tạo sai (`0` thay vì `-∞` cho bài tìm max có giá trị âm…).
- Tối ưu bộ nhớ quá sớm khi chưa chắc công thức đúng — hãy viết bản bảng đầy đủ trước.

## 2.3 Áp dụng vào bài này

Được mua bán nhiều lần → **tham lam**: cộng mọi đoạn tăng `max(0, p[i] − p[i−1])` (lợi nhuận một đoạn tăng dài bằng tổng các bước tăng nhỏ). Cách tổng quát là **DP hai trạng thái** "đang giữ cổ phiếu / không giữ".

# 3. Baseline: thử mọi lựa chọn (đệ quy)

Ở mỗi ngày, trạng thái là "đang cầm cổ phiếu hay không". Thử cả hai hành động hợp lệ:

In [5]:
from typing import List
from functools import lru_cache
import random

def max_profit_brute(prices: List[int]) -> int:
    n = len(prices)
    def go(i, holding):
        if i == n:
            return 0
        best = go(i + 1, holding)                       # không làm gì
        if holding:
            best = max(best, prices[i] + go(i + 1, False))   # bán
        else:
            best = max(best, -prices[i] + go(i + 1, True))   # mua
        return best
    return go(0, False)

print(max_profit_brute([7, 1, 5, 3, 6, 4]), max_profit_brute([1, 2, 3, 4, 5]), max_profit_brute([7, 6, 4, 3, 1]))

7 4 0


`O(2^n)` — chỉ dùng để kiểm tra. Có `2n` trạng thái `(i, holding)` → DP.

# 4. Ý tưởng chính 1: Tham lam — cộng mọi bước tăng

Lợi nhuận của một giao dịch "mua ngày `a`, bán ngày `b`":

```text
p[b] − p[a] = (p[a+1] − p[a]) + (p[a+2] − p[a+1]) + … + (p[b] − p[b−1])
```

tức tổng các **chênh lệch một ngày**. Vì được mua bán không giới hạn (kể cả bán rồi mua lại cùng ngày), ta có thể **chọn riêng từng** chênh lệch một ngày: lấy mọi chênh lệch **dương**, bỏ mọi chênh lệch âm.

```text
lợi nhuận = Σ max(0, p[i] − p[i−1])
```

In [6]:
def max_profit_greedy(prices: List[int]) -> int:
    return sum(max(0, prices[i] - prices[i - 1]) for i in range(1, len(prices)))

print(max_profit_greedy([7, 1, 5, 3, 6, 4]), max_profit_greedy([1, 2, 3, 4, 5]), max_profit_greedy([7, 6, 4, 3, 1]))

7 4 0


# 5. Mô phỏng

In [7]:
def trace_greedy(prices):
    total = 0
    print(f"{'ngày':>4} {'giá':>4} {'chênh':>6}  lấy?  tổng")
    print(f"{0:>4} {prices[0]:>4}")
    for i in range(1, len(prices)):
        d = prices[i] - prices[i - 1]
        take = d > 0
        total += d if take else 0
        print(f"{i:>4} {prices[i]:>4} {d:>+6}  {'có ' if take else 'không'} {total:>4}")
    return total

_ = trace_greedy([7, 1, 5, 3, 6, 4])

ngày  giá  chênh  lấy?  tổng
   0    7
   1    1     -6  không    0
   2    5     +4  có     4
   3    3     -2  không    4
   4    6     +3  có     7
   5    4     -2  không    7


Hai bước tăng `+4` (ngày 1→2) và `+3` (ngày 3→4) ứng đúng với hai giao dịch trong lời giải.

# 6. Ý tưởng chính 2: DP hai trạng thái

Sau ngày `i`:

- `cash` = lợi nhuận lớn nhất khi **không** cầm cổ phiếu.
- `hold` = lợi nhuận lớn nhất khi **đang** cầm cổ phiếu (đã trừ tiền mua).

```text
cash_mới = max(cash, hold + p[i])     # giữ nguyên, hoặc bán hôm nay
hold_mới = max(hold, cash − p[i])     # giữ nguyên, hoặc mua hôm nay
```

Khởi tạo: `cash = 0`, `hold = −∞` (chưa thể cầm cổ phiếu trước ngày 0). Đáp án: `cash` cuối cùng (kết thúc mà còn cầm cổ phiếu thì không lợi).

In [8]:
def max_profit_dp(prices: List[int]) -> int:
    cash, hold = 0, float("-inf")
    for p in prices:
        cash, hold = max(cash, hold + p), max(hold, cash - p)
    return cash

print(max_profit_dp([7, 1, 5, 3, 6, 4]), max_profit_dp([1, 2, 3, 4, 5]), max_profit_dp([7, 6, 4, 3, 1]))

7 4 0


Gán đồng thời `cash, hold = …, …` dùng giá trị **cũ** của cả hai ở vế phải. (Ở bài này dùng giá trị mới cũng không sai — vì mua và bán cùng ngày không tạo lợi nhuận — nhưng với các biến thể có phí hay cooldown thì thứ tự quan trọng.)

In [9]:
def trace_dp(prices):
    cash, hold = 0, float("-inf")
    print(f"{'ngày':>4} {'giá':>4} {'cash':>6} {'hold':>6}")
    for i, p in enumerate(prices):
        cash, hold = max(cash, hold + p), max(hold, cash - p)
        print(f"{i:>4} {p:>4} {cash:>6} {hold:>6}")
    return cash

_ = trace_dp([7, 1, 5, 3, 6, 4])

ngày  giá   cash   hold
   0    7      0     -7
   1    1      0     -1
   2    5      4     -1
   3    3      4      1
   4    6      7      1
   5    4      7      3


# 7. Phiên bản đúng format LeetCode

In [10]:
class Solution:
    def maxProfit(self, prices: List[int]) -> int:
        profit = 0
        for i in range(1, len(prices)):
            if prices[i] > prices[i - 1]:
                profit += prices[i] - prices[i - 1]
        return profit

In [11]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.maxProfit([7, 1, 5, 3, 6, 4]), 7)
check("Ví dụ 2", sol.maxProfit([1, 2, 3, 4, 5]), 4)
check("Ví dụ 3", sol.maxProfit([7, 6, 4, 3, 1]), 0)
check("Một ngày", sol.maxProfit([5]), 0)
check("Giá bằng nhau", sol.maxProfit([3, 3, 3]), 0)

random.seed(122)
for _ in range(3000):
    prices = [random.randint(0, 10) for _ in range(random.randint(1, 12))]
    expected = max_profit_brute(prices)
    assert sol.maxProfit(prices) == expected == max_profit_dp(prices)
print("3000 mảng ngẫu nhiên: tham lam = DP = vét cạn")

Ví dụ 1          PASS  7
Ví dụ 2          PASS  4
Ví dụ 3          PASS  0
Một ngày         PASS  0
Giá bằng nhau    PASS  0


3000 mảng ngẫu nhiên: tham lam = DP = vét cạn


# 8. Vì sao đúng? Độ phức tạp?

**Tham lam là cận trên đạt được:**

- Mọi chiến lược hợp lệ là một tập giao dịch **rời nhau** `[a1, b1], [a2, b2], …` (bán trước khi mua lại). Tổng lợi nhuận = tổng các chênh lệch một ngày nằm **bên trong** các đoạn đó — mỗi chênh lệch một ngày được tính **tối đa một lần**. Vậy tổng ≤ tổng các chênh lệch **dương**.
- Ngược lại, chiến lược "mua ngày `i−1`, bán ngày `i` mỗi khi `p[i] > p[i−1]`" đạt đúng tổng đó (gộp các ngày tăng liên tiếp thành một giao dịch nếu muốn).

**DP:** `cash`/`hold` là quy hoạch động chính xác trên trạng thái `(ngày, đang cầm?)`.

```text
Time:  O(n)
Space: O(1)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: Chỉ tìm một giao dịch tốt nhất (làm thành #121)

In [12]:
def wrong_one_transaction(prices):
    best, lowest = 0, float("inf")
    for p in prices:
        lowest = min(lowest, p)
        best = max(best, p - lowest)              # SAI: chỉ một lần mua-bán
    return best

print("Sai :", wrong_one_transaction([7, 1, 5, 3, 6, 4]), "| Đúng:", sol.maxProfit([7, 1, 5, 3, 6, 4]))

Sai : 5 | Đúng: 7


## Lỗi 2: Mua ở đáy toàn cục, bán ở đỉnh toàn cục

`max(prices) − min(prices)` sai khi đỉnh đứng **trước** đáy, và cũng bỏ lỡ nhiều đoạn tăng.

In [13]:
def wrong_global_extremes(prices):
    return max(0, max(prices) - min(prices))      # SAI

print("Sai :", wrong_global_extremes([7, 6, 4, 3, 1]), "| Đúng:", sol.maxProfit([7, 6, 4, 3, 1]))

Sai : 6 | Đúng: 0


## Lỗi 3: Khởi tạo `hold = 0` trong DP

`hold = 0` nghĩa là "đang cầm cổ phiếu mà không tốn tiền" — có thể bán ở ngày 0 và kiếm lời ảo.

In [14]:
def wrong_hold_zero(prices):
    cash, hold = 0, 0                              # SAI: phải là -inf (hoặc -prices[0])
    for p in prices:
        cash, hold = max(cash, hold + p), max(hold, cash - p)
    return cash

print("Sai :", wrong_hold_zero([5]), "| Đúng:", sol.maxProfit([5]))

Sai : 5 | Đúng: 0


## Lỗi 4: Tìm đỉnh/đáy cục bộ nhưng xử lý sai đoạn giá bằng nhau

Cách "valley–peak" (tìm đáy rồi đỉnh liên tiếp) đúng nhưng dễ sai biên khi giá **đứng yên** (`p[i] == p[i+1]`). Cách cộng chênh lệch dương không có vấn đề này.

# 10. Mẹo nhận diện pattern

**Họ bài cổ phiếu → DP trạng thái** (`cash`, `hold`, có thể thêm số giao dịch đã dùng, cooldown, phí):

| Bài | Ràng buộc | Thay đổi |
|---|---|---|
| #121 | 1 giao dịch | `hold = max(hold, −p)` (mua không dựa vào `cash`) |
| #122 (bài này) | không giới hạn | như mục 6 |
| #714 | phí mỗi giao dịch | `cash = max(cash, hold + p − fee)` |
| #309 | cooldown 1 ngày sau khi bán | mua dựa vào `cash` của **hai ngày trước** |
| #123 / #188 | tối đa 2 / k giao dịch | thêm chiều "số giao dịch" |

Tham lam chỉ đúng khi **không** có chi phí giao dịch và **không** giới hạn số lần.

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Tính lợi nhuận cho `[3, 3, 5, 0, 0, 3, 1, 4]` bằng cả hai cách.

## Bài 2 — Điền chỗ trống (DP)

In [15]:
def exercise_2_max_profit(prices: List[int]) -> int:
    cash, hold = 0, float("-inf")
    for p in prices:
        # TODO: cập nhật đồng thời cash và hold
        pass
    return cash


# assert exercise_2_max_profit([7, 1, 5, 3, 6, 4]) == 7
# assert exercise_2_max_profit([1, 2, 3, 4, 5]) == 4
# assert exercise_2_max_profit([7, 6, 4, 3, 1]) == 0

## Bài 3 — In các giao dịch

Trả về list các cặp `(ngày mua, ngày bán)` của một chiến lược tối ưu với **ít giao dịch nhất** (gộp các ngày tăng liên tiếp).

```python
transactions([7, 1, 5, 3, 6, 4]) -> [(1, 2), (3, 4)]
transactions([1, 2, 3, 4, 5])    -> [(0, 4)]
```

In [16]:
def exercise_3_transactions(prices: List[int]):
    # TODO
    pass


# assert exercise_3_transactions([7, 1, 5, 3, 6, 4]) == [(1, 2), (3, 4)]
# assert exercise_3_transactions([1, 2, 3, 4, 5]) == [(0, 4)]
# assert exercise_3_transactions([7, 6, 4, 3, 1]) == []
# assert exercise_3_transactions([1, 2, 2, 3]) == [(0, 1), (2, 3)]

## Bài 4 — Biến thể: có phí giao dịch (LeetCode #714)

Mỗi lần **bán** phải trả phí `fee`. Lợi nhuận lớn nhất?

```python
max_profit_fee([1, 3, 2, 8, 4, 9], 2)  -> 8
max_profit_fee([1, 3, 7, 5, 10, 3], 3) -> 6
```

In [17]:
def exercise_4_max_profit_fee(prices: List[int], fee: int) -> int:
    # TODO
    pass


# assert exercise_4_max_profit_fee([1, 3, 2, 8, 4, 9], 2) == 8
# assert exercise_4_max_profit_fee([1, 3, 7, 5, 10, 3], 3) == 6

## Bài 5 — Mini challenge: cooldown (LeetCode #309)

Sau khi bán, phải nghỉ **một ngày** mới được mua lại.

```python
max_profit_cooldown([1, 2, 3, 0, 2]) -> 3     # mua, bán, nghỉ, mua, bán
max_profit_cooldown([1])             -> 0
```

Gợi ý nhanh: ba trạng thái `hold`, `sold` (vừa bán hôm nay), `rest` (không cầm, không bị cấm mua).

In [18]:
def exercise_5_max_profit_cooldown(prices: List[int]) -> int:
    # TODO
    pass


# assert exercise_5_max_profit_cooldown([1, 2, 3, 0, 2]) == 3
# assert exercise_5_max_profit_cooldown([1]) == 0
# assert exercise_5_max_profit_cooldown([2, 1, 4]) == 3

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `trace_greedy` và `trace_dp` sau khi tự làm.

**Bài 3**
- Mức 1: duyệt `i`; khi `p[i] > p[i−1]` mà chưa mở giao dịch thì mở ở `i − 1`; khi `p[i] <= p[i−1]` mà đang mở thì đóng ở `i − 1`.
- Mức 2: nhớ đóng giao dịch đang mở ở cuối mảng.

**Bài 4**
- Mức 1: như DP mục 6, trừ `fee` khi bán.

**Bài 5**
- Mức 1: `hold' = max(hold, rest − p)`; `sold' = hold + p`; `rest' = max(rest, sold)`.
- Mức 2: khởi tạo `hold = −∞`, `sold = −∞`, `rest = 0`; đáp án `max(sold, rest)`.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [19]:
_ = trace_greedy([3, 3, 5, 0, 0, 3, 1, 4])
print()
_ = trace_dp([3, 3, 5, 0, 0, 3, 1, 4])

ngày  giá  chênh  lấy?  tổng
   0    3
   1    3     +0  không    0
   2    5     +2  có     2
   3    0     -5  không    2
   4    0     +0  không    2
   5    3     +3  có     5
   6    1     -2  không    5
   7    4     +3  có     8

ngày  giá   cash   hold
   0    3      0     -3
   1    3      0     -3
   2    5      2     -3
   3    0      2      2
   4    0      2      2
   5    3      5      2
   6    1      5      4
   7    4      8      4


## Đáp án Bài 2

In [20]:
def answer_2_max_profit(prices: List[int]) -> int:
    return max_profit_dp(prices)

assert answer_2_max_profit([7, 1, 5, 3, 6, 4]) == 7
assert answer_2_max_profit([1, 2, 3, 4, 5]) == 4
assert answer_2_max_profit([7, 6, 4, 3, 1]) == 0
print("Bài 2 PASS (giống max_profit_dp ở mục 6)")

Bài 2 PASS (giống max_profit_dp ở mục 6)


## Đáp án Bài 3

In [21]:
def answer_3_transactions(prices: List[int]):
    out, buy = [], None
    for i in range(1, len(prices)):
        if prices[i] > prices[i - 1]:
            if buy is None:
                buy = i - 1
        elif buy is not None:
            out.append((buy, i - 1))
            buy = None
    if buy is not None:
        out.append((buy, len(prices) - 1))
    return out

assert answer_3_transactions([7, 1, 5, 3, 6, 4]) == [(1, 2), (3, 4)]
assert answer_3_transactions([1, 2, 3, 4, 5]) == [(0, 4)]
assert answer_3_transactions([7, 6, 4, 3, 1]) == []
assert answer_3_transactions([1, 2, 2, 3]) == [(0, 1), (2, 3)]
for _ in range(2000):
    prices = [random.randint(0, 10) for _ in range(random.randint(1, 12))]
    tr = answer_3_transactions(prices)
    assert sum(prices[b] - prices[a] for a, b in tr) == sol.maxProfit(prices)
    assert all(a < b for a, b in tr) and all(tr[k][1] < tr[k + 1][0] for k in range(len(tr) - 1))
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [22]:
def answer_4_max_profit_fee(prices: List[int], fee: int) -> int:
    cash, hold = 0, float("-inf")
    for p in prices:
        cash, hold = max(cash, hold + p - fee), max(hold, cash - p)
    return cash

def brute_fee(prices, fee):
    @lru_cache(maxsize=None)
    def go(i, h):
        if i == len(prices):
            return 0
        best = go(i + 1, h)
        best = max(best, (prices[i] - fee + go(i + 1, False)) if h else (-prices[i] + go(i + 1, True)))
        return best
    return go(0, False)

assert answer_4_max_profit_fee([1, 3, 2, 8, 4, 9], 2) == 8
assert answer_4_max_profit_fee([1, 3, 7, 5, 10, 3], 3) == 6
for _ in range(2000):
    prices = [random.randint(0, 10) for _ in range(random.randint(1, 10))]
    fee = random.randint(0, 4)
    assert answer_4_max_profit_fee(prices, fee) == brute_fee(tuple(prices), fee)
print("Bài 4 PASS")

Bài 4 PASS


Có phí thì tham lam "cộng mọi bước tăng" **sai**: mỗi bước tăng nhỏ hơn phí sẽ gây lỗ — đó là lý do DP là khuôn mẫu tổng quát hơn.

## Đáp án Bài 5

In [23]:
def answer_5_max_profit_cooldown(prices: List[int]) -> int:
    hold, sold, rest = float("-inf"), float("-inf"), 0
    for p in prices:
        hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
    return max(sold, rest)

def brute_cooldown(prices):
    @lru_cache(maxsize=None)
    def go(i, h):
        if i >= len(prices):
            return 0
        best = go(i + 1, h)
        if h:
            best = max(best, prices[i] + go(i + 2, False))      # bán, nghỉ ngày i+1
        else:
            best = max(best, -prices[i] + go(i + 1, True))
        return best
    return go(0, False)

assert answer_5_max_profit_cooldown([1, 2, 3, 0, 2]) == 3
assert answer_5_max_profit_cooldown([1]) == 0
assert answer_5_max_profit_cooldown([2, 1, 4]) == 3
for _ in range(2000):
    prices = tuple(random.randint(0, 10) for _ in range(random.randint(1, 10)))
    assert answer_5_max_profit_cooldown(list(prices)) == brute_cooldown(prices)
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> Không giới hạn giao dịch, không phí: lợi nhuận tối đa = **tổng mọi chênh lệch dương** giữa hai ngày liên tiếp. Khuôn mẫu tổng quát: DP `cash = max(cash, hold + p)`, `hold = max(hold, cash − p)`, khởi tạo `hold = −∞`.

---

## Checklist trước khi nộp

- [ ] Không nhầm với #121 (một giao dịch)?
- [ ] DP: `hold` khởi tạo `−∞`?
- [ ] Đáp án là `cash` (không cầm cổ phiếu cuối cùng)?
- [ ] Biết khi nào tham lam không còn đúng (phí, cooldown, giới hạn số giao dịch)?